In [1]:
# ================================================================
# NOTEBOOK 13: CWQ QWEN GAIN CONTROLLER TRAINING
# Runtime: T4 GPU
# No dataset download. No test-set evaluation.
# ================================================================

import collections
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import math
import os
import random
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np


# ---------------------------- SETTINGS ----------------------------

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

SEED = 42
PER_CLASS = 15000
CHECKPOINT_DEV_SIZE = 8000

MAX_LENGTH = 256
TRAIN_BATCH = 4
GRAD_ACCUM = 8
INFERENCE_BATCH = 64

EPOCHS = 1
LEARNING_RATE = 2e-4
EVAL_STEPS = 500
PREDICTION_CHUNK_STATES = 64

CODE_VERSION = "cwq_gain_training_v1_0"

CLASSES = ["0", "1", "2", "3_plus"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}


# ------------------------- FILE UTILITIES -------------------------

def fingerprint(obj):
    serialized = json.dumps(obj, sort_keys=True).encode()
    return hashlib.sha256(serialized).hexdigest()


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, obj):
    temporary = path.with_name(path.name + ".partial")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            obj,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
    temporary.replace(path)


def rows(path):
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip():
                yield json.loads(line)


# --------------------------- INPUT AUDIT --------------------------

def audit_row(row, split, allowed_qids):
    if row["split"] != split or row["qid"] not in allowed_qids:
        raise ValueError("Wrong question or split in the state data.")

    if row["candidate_count"] != len(row["candidates"]):
        raise ValueError("Candidate count mismatch.")

    # Global STOP truth comes from the FULL frontier.
    if row["stop_label"] != int(row["full_positive_count"] == 0):
        raise ValueError("Incorrect full-frontier STOP target.")

    if split == "dev":
        if (
            row["candidates_sampled"]
            or len(row["candidates"]) != row["shortlist_candidate_count"]
        ):
            raise ValueError(
                "Calibration requires every development shortlist candidate."
            )

    history = (
        " -> ".join(row["history_relations"][-6:])
        if row["history_relations"]
        else "none"
    )

    for candidate in row["candidates"]:
        expected_text = (
            f'Question: {row["question"]}\n'
            f'Answers found so far: {row["found_count"]}\n'
            f'Visited entities: {row["visited_count"]}\n'
            f'Search step: {row["step"]}\n'
            f'Source depth: {candidate["source_depth"]}\n'
            f'Candidate relation: {candidate["relation_text"]}\n'
            f'Candidate target count: {candidate["target_count"]}\n'
            f"Recent relations: {history}"
        )

        if (
            candidate["model_text"] != expected_text
            or candidate["gain_class"] not in CLASS_TO_ID
        ):
            raise ValueError("Invalid controller input or gain class.")

        expected_class = (
            "3_plus"
            if candidate["gain"] >= 3
            else str(candidate["gain"])
        )

        if candidate["gain_class"] != expected_class:
            raise ValueError("Gain and gain-class disagreement.")


# ----------------------- RESERVOIR SAMPLING -----------------------

def sampled_candidates(
    path,
    split,
    expected,
    allowed,
    cache_dir,
):
    specification = {
        "code_version": CODE_VERSION,
        "split": split,
        "source_sha256": expected["sha256"],
        "seed": SEED,
        "per_class": PER_CLASS,
        "checkpoint_dev_size": CHECKPOINT_DEV_SIZE,
    }

    signature = fingerprint(specification)

    cache_path = cache_dir / f"{split}_sample_v1.jsonl"
    metadata_path = cache_dir / f"{split}_sample_v1.json"

    metadata = (
        read_json(metadata_path)
        if metadata_path.exists() and cache_path.exists()
        else None
    )

    if (
        metadata
        and metadata.get("signature") == signature
        and metadata.get("sha256") == file_hash(cache_path)
    ):
        sample = list(rows(cache_path))

        if len(sample) != metadata["sampled_total"]:
            raise ValueError("Cached sample length mismatch.")

        print(
            f"{split}: reused {len(sample):,} cached candidates.",
            flush=True,
        )
        return sample, metadata

    seen = collections.Counter()
    reservoirs = {name: [] for name in CLASSES}

    class_rngs = {
        name: random.Random(SEED + 100 + i)
        for i, name in enumerate(CLASSES)
    }

    dev_reservoir = []
    dev_rng = random.Random(SEED + 999)

    state_count = 0
    candidate_count = 0

    for row in rows(path):
        audit_row(row, split, allowed)
        state_count += 1

        for candidate in row["candidates"]:
            name = candidate["gain_class"]
            seen[name] += 1
            candidate_count += 1

            item = {
                "text": candidate["model_text"],
                "label": CLASS_TO_ID[name],
                "qid": row["qid"],
                "state_id": row["state_id"],
            }

            if split == "train":
                bucket = reservoirs[name]
                rng = class_rngs[name]

                if len(bucket) < PER_CLASS:
                    bucket.append(item)
                else:
                    index = rng.randrange(seen[name])
                    if index < PER_CLASS:
                        bucket[index] = item

            else:
                # Natural, uniform development sample.
                # Development data are not class-balanced.
                if len(dev_reservoir) < CHECKPOINT_DEV_SIZE:
                    dev_reservoir.append(item)
                else:
                    index = dev_rng.randrange(candidate_count)
                    if index < CHECKPOINT_DEV_SIZE:
                        dev_reservoir[index] = item

        if state_count % 5000 == 0:
            print(
                f"{split}: {state_count:,} states / "
                f"{candidate_count:,} candidates scanned",
                flush=True,
            )

    if (
        state_count != expected["states"]
        or candidate_count != expected["candidates"]
    ):
        raise ValueError(
            "Source totals disagree with Notebook 12."
        )

    if dict(seen) != expected["gain_classes"]:
        raise ValueError(
            "Source gain-class totals disagree with Notebook 12."
        )

    if split == "train":
        if any(
            len(reservoirs[name]) != PER_CLASS
            for name in CLASSES
        ):
            raise ValueError(
                "Insufficient examples for the balanced training sample."
            )

        sample = [
            item
            for name in CLASSES
            for item in reservoirs[name]
        ]
    else:
        sample = dev_reservoir

    random.Random(SEED).shuffle(sample)

    temporary = cache_path.with_name(cache_path.name + ".partial")

    with open(temporary, "w", encoding="utf-8") as file:
        for item in sample:
            file.write(
                json.dumps(item, ensure_ascii=False) + "\n"
            )

    temporary.replace(cache_path)

    distribution = collections.Counter(
        CLASSES[item["label"]] for item in sample
    )

    metadata = {
        "signature": signature,
        "specification": specification,
        "states_read": state_count,
        "candidates_read": candidate_count,
        "observed_class_counts": dict(seen),
        "sampled_class_counts": dict(distribution),
        "sampled_total": len(sample),
        "sampled_questions": len(
            {item["qid"] for item in sample}
        ),
        "sha256": file_hash(cache_path),
    }

    save_json(metadata_path, metadata)

    return sample, metadata


# ----------------------- CANDIDATE METRICS ------------------------

def metrics_from_cm(matrix):
    cm = np.asarray(matrix, dtype=np.int64)

    total = int(cm.sum())
    true_positive = np.diag(cm).astype(float)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)

    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros(4),
        where=predicted > 0,
    )

    recall = np.divide(
        true_positive,
        support,
        out=np.zeros(4),
        where=support > 0,
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(4),
        where=(precision + recall) > 0,
    )

    return {
        "number_of_examples": total,
        "accuracy": float(
            true_positive.sum() / max(1, total)
        ),
        "balanced_accuracy": (
            float(recall[support > 0].mean())
            if np.any(support > 0)
            else 0.0
        ),
        "macro_f1": float(f1.mean()),
        "weighted_f1": float(
            np.dot(f1, support) / max(1, total)
        ),
        "class_distribution": {
            CLASSES[i]: int(support[i])
            for i in range(4)
        },
        "classification_report": {
            f"gain_{CLASSES[i]}": {
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i in range(4)
        },
        "confusion_matrix": cm.tolist(),
    }


def candidate_cm(labels, predictions):
    labels = np.asarray(labels, dtype=int)
    predictions = np.asarray(predictions, dtype=int)

    return np.bincount(
        4 * labels + predictions,
        minlength=16,
    ).reshape(4, 4)


# --------------------- DEVELOPMENT PREDICTIONS --------------------

def prediction_chunk(state_rows, predictor):
    texts = []
    labels = []
    lengths = []

    for row in state_rows:
        lengths.append(len(row["candidates"]))

        for candidate in row["candidates"]:
            texts.append(candidate["model_text"])
            labels.append(
                CLASS_TO_ID[candidate["gain_class"]]
            )

    probabilities = (
        np.asarray(predictor(texts), dtype=np.float32)
        if texts
        else np.empty((0, 4), dtype=np.float32)
    )

    if (
        probabilities.shape != (len(texts), 4)
        or not np.isfinite(probabilities).all()
    ):
        raise ValueError("Invalid candidate probabilities.")

    if len(probabilities):
        if not np.allclose(
            probabilities.sum(axis=1),
            1.0,
            atol=1e-4,
        ):
            raise ValueError(
                "Candidate probabilities do not sum to one."
            )

    labels_array = np.asarray(labels, dtype=int)

    predictions = (
        probabilities.argmax(axis=1)
        if len(probabilities)
        else np.array([], dtype=int)
    )

    cm = candidate_cm(labels_array, predictions)

    confidence = (
        probabilities.max(axis=1)
        if len(probabilities)
        else np.array([])
    )

    bins = np.minimum(
        (confidence * 15).astype(int),
        14,
    )

    correct = predictions == labels_array

    ece_counts = np.bincount(bins, minlength=15)
    ece_confidence = np.bincount(
        bins,
        weights=confidence,
        minlength=15,
    )
    ece_correct = np.bincount(
        bins,
        weights=correct.astype(float),
        minlength=15,
    )

    brier_sum = (
        float(
            (
                np.square(probabilities).sum(axis=1)
                - 2 * probabilities[
                    np.arange(len(labels_array)),
                    labels_array,
                ]
                + 1
            ).sum()
        )
        if len(labels_array)
        else 0.0
    )

    records = []
    offset = 0

    for row, size in zip(state_rows, lengths):
        group = probabilities[offset:offset + size]

        # max P(gain > 0) across shortlist candidates.
        continue_score = (
            1.0 - float(group[:, 0].min())
            if size
            else 0.0
        )

        records.append({
            "state_id": row["state_id"],
            "qid": row["qid"],
            "trajectory_policy": row["trajectory_policy"],
            "step": row["step"],
            "candidate_count": size,
            "max_continue_score": continue_score,
            "stop_score_proxy": 1.0 - continue_score,
            "global_stop_label": row["stop_label"],
            "shortlist_stop_label": row["shortlist_stop_label"],
            "shortlist_hides_all_positive": (
                row["shortlist_hides_all_positive"]
            ),
        })

        offset += size

    return {
        "candidate_count": len(texts),
        "confusion_matrix": cm.tolist(),
        "brier_sum": brier_sum,
        "ece_counts": ece_counts.tolist(),
        "ece_conf_sum": ece_confidence.tolist(),
        "ece_correct_sum": ece_correct.tolist(),
        "state_records": records,
    }


def stop_metrics(records, threshold):
    truth = np.array(
        [row["global_stop_label"] for row in records],
        dtype=bool,
    )

    scores = np.array(
        [row["max_continue_score"] for row in records],
        dtype=float,
    )

    empty = np.array(
        [row["candidate_count"] == 0 for row in records],
        dtype=bool,
    )

    predicted_stop = empty | (scores < threshold)

    tp = int(np.sum(truth & predicted_stop))
    fp = int(np.sum(~truth & predicted_stop))
    fn = int(np.sum(truth & ~predicted_stop))
    tn = int(np.sum(~truth & ~predicted_stop))

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    return {
        "threshold": float(threshold),
        "stop_precision": precision,
        "stop_recall": recall,
        "stop_f1": (
            2 * precision * recall
            / max(1e-15, precision + recall)
        ),
        "accuracy": (
            (tp + tn) / max(1, len(records))
        ),
        "true_stop_states": tp + fn,
        "true_continue_states": fp + tn,
        "false_stop_count": fp,
        "false_continue_count": fn,
        "premature_stop_rate_given_continue": (
            fp / max(1, fp + tn)
        ),
        "premature_stop_fraction_all_states": (
            fp / max(1, len(records))
        ),
        "over_search_rate_given_stop": (
            fn / max(1, tp + fn)
        ),
        "over_search_fraction_all_states": (
            fn / max(1, len(records))
        ),
    }


def calibrate(records):
    sweep = [
        stop_metrics(records, threshold)
        for threshold in np.linspace(0.0, 1.0, 1001)
    ]

    best = min(
        sweep,
        key=lambda result: (
            -result["stop_f1"],
            result["premature_stop_rate_given_continue"],
            -result["accuracy"],
            result["threshold"],
        ),
    )

    return best, sweep


def complete_dev_predictions(
    path,
    expected,
    allowed,
    output,
    signature,
    predictor,
):
    chunks_dir = (
        output
        / "dev_prediction_chunks"
        / signature[:12]
    )
    chunks_dir.mkdir(parents=True, exist_ok=True)

    totals = np.zeros((4, 4), dtype=np.int64)
    ece_counts = np.zeros(15)
    ece_confidence = np.zeros(15)
    ece_correct = np.zeros(15)

    all_records = []
    brier_total = 0.0
    candidate_total = 0
    buffer = []
    chunk_index = 0
    seen_ids = set()

    started = time.time()

    def process(group, index):
        path_chunk = chunks_dir / f"chunk_{index:04d}.json"
        state_ids = [row["state_id"] for row in group]

        cached = (
            read_json(path_chunk)
            if path_chunk.exists()
            else None
        )

        if (
            cached
            and cached.get("signature") == signature
            and cached.get("state_ids") == state_ids
        ):
            return cached

        result = prediction_chunk(group, predictor)
        result.update(
            signature=signature,
            state_ids=state_ids,
        )

        save_json(path_chunk, result)
        return result

    def accumulate(result):
        nonlocal totals
        nonlocal ece_counts
        nonlocal ece_confidence
        nonlocal ece_correct
        nonlocal brier_total
        nonlocal candidate_total

        totals += np.asarray(
            result["confusion_matrix"],
            dtype=np.int64,
        )
        ece_counts += np.asarray(result["ece_counts"])
        ece_confidence += np.asarray(
            result["ece_conf_sum"]
        )
        ece_correct += np.asarray(
            result["ece_correct_sum"]
        )

        brier_total += result["brier_sum"]
        candidate_total += result["candidate_count"]

        all_records.extend(result["state_records"])

    for row in rows(path):
        audit_row(row, "dev", allowed)

        if row["state_id"] in seen_ids:
            raise ValueError("Duplicate development state ID.")

        seen_ids.add(row["state_id"])
        buffer.append(row)

        if len(buffer) == PREDICTION_CHUNK_STATES:
            accumulate(process(buffer, chunk_index))
            buffer = []
            chunk_index += 1

            if chunk_index % 8 == 0:
                print(
                    f'Full dev: {len(all_records):,}/'
                    f'{expected["states"]:,} states | '
                    f"{candidate_total:,} candidates | "
                    f"{(time.time() - started) / 60:.1f} min",
                    flush=True,
                )

    if buffer:
        accumulate(process(buffer, chunk_index))

    if (
        len(all_records) != expected["states"]
        or candidate_total != expected["candidates"]
    ):
        raise ValueError(
            "Full development totals disagree with Notebook 12."
        )

    metrics = metrics_from_cm(totals)

    expected_distribution = {
        name: expected["gain_classes"][name]
        for name in CLASSES
    }

    if metrics["class_distribution"] != expected_distribution:
        raise ValueError(
            "Full development class totals disagree with Notebook 12."
        )

    metrics["majority_class_accuracy"] = (
        max(expected["gain_classes"].values())
        / candidate_total
    )

    metrics["multiclass_brier_sum_over_classes"] = (
        brier_total / max(1, candidate_total)
    )

    metrics["top_label_ece_15_bins"] = float(
        np.abs(ece_confidence - ece_correct).sum()
        / max(1, candidate_total)
    )

    metrics["states"] = len(all_records)
    metrics["runtime_minutes_this_invocation"] = (
        time.time() - started
    ) / 60

    return metrics, all_records


# ----------------------------- MAIN ------------------------------

def main():
    from google.colab import drive
    drive.mount("/content/drive")

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Select Runtime > Change runtime type > T4 GPU, "
            "then rerun this cell."
        )

    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"

    requirements = [
        "transformers>=4.57,<6",
        "peft>=0.18,<1",
        "accelerate>=1.12,<2",
        "bitsandbytes>=0.49,<1",
        "safetensors>=0.6,<1",
        "tqdm>=4.60",
    ]

    # Reuse compatible installations and preserve Colab's PyTorch.
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
    ) as constraint:
        constraint.write(
            "torch==" + torch.__version__.split("+")[0] + "\n"
        )
        constraint.flush()

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-c",
                constraint.name,
                *requirements,
            ],
            check=True,
        )

    importlib.invalidate_caches()

    import transformers
    import peft

    from transformers import (
        AutoConfig,
        AutoTokenizer,
        AutoModelForSequenceClassification,
        BitsAndBytesConfig,
        TrainingArguments,
        Trainer,
        TrainerCallback,
    )

    from peft import (
        LoraConfig,
        TaskType,
        get_peft_model,
        prepare_model_for_kbit_training,
        PeftModel,
    )

    from safetensors import safe_open
    from tqdm.auto import tqdm

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    preprocessing_dir = ROOT / "outputs/cwq_preprocessing_v1"
    output_dir = ROOT / "outputs/cwq_controller_training_v1"

    cache_dir = (
        ROOT / "data/processed/cwq_controller_training_cache_v1"
    )

    run_dir = ROOT / "models/cwq_gain_controller_training_v1"
    adapter_dir = (
        ROOT / "models/cwq_setstop_qwen_gain_controller_v1"
    )

    for path in [output_dir, cache_dir, run_dir]:
        path.mkdir(parents=True, exist_ok=True)

    generation = read_json(
        preprocessing_dir / "cwq_state_generation_summary_v1.json"
    )
    audit = read_json(
        preprocessing_dir / "cwq_state_integrity_audit_v1.json"
    )
    inventory = read_json(
        preprocessing_dir / "artifact_inventory_v1.json"
    )
    selection = read_json(
        preprocessing_dir / "cwq_frontier_selection_v1.json"
    )

    if not (
        generation["signature"]
        == audit["signature"]
        == inventory["signature"]
        and audit["passed"]
    ):
        raise ValueError(
            "Notebook-12 integrity or signature checks failed."
        )

    report_names = [
        "cwq_state_generation_summary_v1.json",
        "cwq_state_integrity_audit_v1.json",
        "cwq_frontier_selection_v1.json",
    ]

    for name in report_names:
        item = next(
            entry
            for entry in inventory["files"]
            if entry["name"] == name
        )

        if file_hash(preprocessing_dir / name) != item["sha256"]:
            raise ValueError(
                f"Notebook-12 report hash mismatch: {name}"
            )

    configuration = generation["configuration"]

    if (
        int(selection["selected_frontier_limit"])
        != configuration["frontier_limit"]
    ):
        raise ValueError(
            "Frontier selection does not match generated states."
        )

    allowed_qids = {
        split: set(configuration["selected_qids"][split])
        for split in ["train", "dev"]
    }

    if allowed_qids["train"] & allowed_qids["dev"]:
        raise ValueError("Train/development question overlap.")

    print("=" * 90)
    print("NOTEBOOK 13: CWQ QWEN GAIN CONTROLLER TRAINING")
    print("=" * 90)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Transformers:", transformers.__version__)
    print("PyTorch:", torch.__version__)
    print("PEFT:", peft.__version__)
    print("Frontier limit:", configuration["frontier_limit"])
    print("Action cap:", configuration["action_cap"])
    print("Maximum hops:", configuration["max_hops"])

    state_paths = {
        split: Path(generation["splits"][split]["path"])
        for split in ["train", "dev"]
    }

    for split, path in state_paths.items():
        expected = generation["splits"][split]

        print("Verifying state file:", path.name, flush=True)

        if (
            path.stat().st_size != expected["bytes"]
            or file_hash(path) != expected["sha256"]
        ):
            raise ValueError(
                f"State file changed since Notebook 12: {split}"
            )

    train_sample, train_stats = sampled_candidates(
        state_paths["train"],
        "train",
        generation["splits"]["train"],
        allowed_qids["train"],
        cache_dir,
    )

    dev_sample, dev_stats = sampled_candidates(
        state_paths["dev"],
        "dev",
        generation["splits"]["dev"],
        allowed_qids["dev"],
        cache_dir,
    )

    print(
        "Training sample:",
        json.dumps(train_stats["sampled_class_counts"]),
    )
    print(
        "Checkpoint development sample:",
        json.dumps(dev_stats["sampled_class_counts"]),
    )
    print(
        "Full development calibration candidates:",
        generation["splits"]["dev"]["candidates"],
    )

    specification = {
        "code_version": CODE_VERSION,
        "data_signature": generation["signature"],
        "model": MODEL_NAME,
        "seed": SEED,
        "per_class": PER_CLASS,
        "checkpoint_dev_size": CHECKPOINT_DEV_SIZE,
        "max_length": MAX_LENGTH,
        "train_batch": TRAIN_BATCH,
        "gradient_accumulation": GRAD_ACCUM,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lora_rank": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
        "class_to_id": CLASS_TO_ID,
        "train_sample_sha256": train_stats["sha256"],
        "dev_sample_sha256": dev_stats["sha256"],
    }

    specification_signature = fingerprint(specification)

    manifest_path = run_dir / "run_manifest_v1.json"

    run_manifest = (
        read_json(manifest_path)
        if manifest_path.exists()
        else None
    )

    if (
        run_manifest
        and run_manifest["spec_signature"]
        != specification_signature
    ):
        raise ValueError(
            "The existing CWQ run has different inputs or settings. "
            "Use a new run directory for a different experiment."
        )

    hf_cache = "/content/setstop_cwq_hf_cache_v1"

    revision = (
        run_manifest["base_revision"]
        if run_manifest
        else None
    )

    base_config = AutoConfig.from_pretrained(
        MODEL_NAME,
        revision=revision or "main",
        token=False,
        cache_dir=hf_cache,
    )

    revision = revision or base_config._commit_hash

    if not revision:
        raise ValueError(
            "Could not resolve the base-model revision."
        )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        revision=revision,
        token=False,
        cache_dir=hf_cache,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    if run_manifest is None:
        run_manifest = {
            "spec_signature": specification_signature,
            "specification": specification,
            "base_revision": revision,
        }
        save_json(manifest_path, run_manifest)

    class CandidateDataset(torch.utils.data.Dataset):
        def __init__(self, examples):
            self.ids = []
            self.labels = []

            for start in tqdm(
                range(0, len(examples), 512),
                desc="Tokenizing candidates",
            ):
                batch = examples[start:start + 512]

                encoded = tokenizer(
                    [item["text"] for item in batch],
                    truncation=True,
                    max_length=MAX_LENGTH,
                    padding=False,
                )

                self.ids.extend(encoded["input_ids"])
                self.labels.extend(
                    item["label"] for item in batch
                )

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            return {
                "input_ids": self.ids[index],
                "labels": self.labels[index],
            }

    def collate(features):
        batch = tokenizer.pad(
            [
                {"input_ids": item["input_ids"]}
                for item in features
            ],
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )

        batch["labels"] = torch.tensor(
            [item["labels"] for item in features],
            dtype=torch.long,
        )

        return batch

    train_dataset = CandidateDataset(train_sample)
    dev_dataset = CandidateDataset(dev_sample)

    complete_path = run_dir / "training_complete_v1.json"

    completed = (
        read_json(complete_path)
        if complete_path.exists()
        else None
    )

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=["score"],
    )

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        revision=revision,
        num_labels=4,
        id2label={
            i: f"gain_{name}"
            for i, name in enumerate(CLASSES)
        },
        label2id={
            f"gain_{name}": i
            for i, name in enumerate(CLASSES)
        },
        problem_type="single_label_classification",
        quantization_config=quantization,
        dtype=torch.float16,
        device_map={"": 0},
        token=False,
        cache_dir=hf_cache,
        attn_implementation="sdpa",
    )

    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.use_cache = False

    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        },
    )

    if completed:
        if (
            completed["spec_signature"]
            != specification_signature
            or file_hash(
                adapter_dir / "adapter_model.safetensors"
            ) != completed["adapter_sha256"]
        ):
            raise ValueError(
                "Completed adapter does not match this run."
            )

        model = PeftModel.from_pretrained(
            base_model,
            str(adapter_dir),
            is_trainable=False,
        )

        print(
            "Reusing completed CWQ adapter. "
            "Training will not repeat."
        )

        save_json(
            output_dir / "cwq_training_metrics_v1.json",
            completed,
        )

    else:
        lora_configuration = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
            ],
            modules_to_save=["score"],
            bias="none",
            revision=revision,
        )

        model = get_peft_model(
            base_model,
            lora_configuration,
        )

        # AMP requires trainable parameters to remain FP32.
        for parameter in model.parameters():
            if parameter.requires_grad:
                parameter.data = parameter.data.float()

        if not any(
            "score" in name and parameter.requires_grad
            for name, parameter in model.named_parameters()
        ):
            raise ValueError(
                "The classification head is not trainable."
            )

        model.print_trainable_parameters()

        optimizer_steps = (
            math.ceil(
                math.ceil(len(train_dataset) / TRAIN_BATCH)
                / GRAD_ACCUM
            )
            * EPOCHS
        )

        argument_values = {
            "output_dir": str(run_dir / "checkpoints"),
            "num_train_epochs": EPOCHS,
            "per_device_train_batch_size": TRAIN_BATCH,
            "per_device_eval_batch_size": INFERENCE_BATCH,
            "gradient_accumulation_steps": GRAD_ACCUM,
            "learning_rate": LEARNING_RATE,

            # Compatible with the installed Transformers API.
            "warmup_steps": math.ceil(0.1 * optimizer_steps),

            "weight_decay": 0.01,
            "lr_scheduler_type": "linear",
            "fp16": True,
            "bf16": False,
            "gradient_checkpointing": True,
            "gradient_checkpointing_kwargs": {
                "use_reentrant": False,
            },
            "max_grad_norm": 1.0,
            "eval_steps": EVAL_STEPS,
            "save_strategy": "steps",
            "save_steps": EVAL_STEPS,
            "save_total_limit": 2,
            "load_best_model_at_end": True,
            "metric_for_best_model": "macro_f1",
            "greater_is_better": True,
            "logging_steps": 50,
            "report_to": [],
            "optim": "adamw_torch",
            "seed": SEED,
            "data_seed": SEED,
            "remove_unused_columns": False,
            "label_names": ["labels"],
            "dataloader_num_workers": 0,
            "save_safetensors": True,
            "eval_accumulation_steps": 16,
        }

        supported = inspect.signature(
            TrainingArguments
        ).parameters

        evaluation_key = (
            "eval_strategy"
            if "eval_strategy" in supported
            else "evaluation_strategy"
        )

        argument_values[evaluation_key] = "steps"

        essential_arguments = [
            "output_dir",
            "num_train_epochs",
            "per_device_train_batch_size",
            "gradient_accumulation_steps",
            "learning_rate",
            "warmup_steps",
            "fp16",
            evaluation_key,
            "load_best_model_at_end",
            "metric_for_best_model",
            "save_steps",
            "eval_steps",
            "label_names",
        ]

        if any(
            name not in supported
            for name in essential_arguments
        ):
            raise RuntimeError(
                "Installed TrainingArguments lacks "
                "a required training option."
            )

        ignored = [
            name
            for name in argument_values
            if name not in supported
        ]

        if ignored:
            print(
                "Unsupported optional arguments omitted:",
                ignored,
            )

        training_arguments = TrainingArguments(
            **{
                name: value
                for name, value in argument_values.items()
                if name in supported
            }
        )

        def compute_metrics(prediction):
            logits = prediction.predictions

            if isinstance(logits, tuple):
                logits = logits[0]

            metrics = metrics_from_cm(
                candidate_cm(
                    prediction.label_ids,
                    np.argmax(logits, axis=1),
                )
            )

            return {
                name: metrics[name]
                for name in [
                    "accuracy",
                    "balanced_accuracy",
                    "macro_f1",
                    "weighted_f1",
                ]
            }

        class FinalStepCallback(TrainerCallback):
            def on_step_end(
                self,
                args,
                state,
                control,
                **kwargs,
            ):
                if state.global_step == state.max_steps:
                    control.should_evaluate = True
                    control.should_save = True

                return control

        trainer_values = {
            "model": model,
            "args": training_arguments,
            "train_dataset": train_dataset,
            "eval_dataset": dev_dataset,
            "data_collator": collate,
            "compute_metrics": compute_metrics,
            "callbacks": [FinalStepCallback()],
        }

        trainer_parameters = inspect.signature(
            Trainer.__init__
        ).parameters

        if "processing_class" in trainer_parameters:
            trainer_values["processing_class"] = tokenizer
        elif "tokenizer" in trainer_parameters:
            trainer_values["tokenizer"] = tokenizer

        trainer = Trainer(**trainer_values)

        # The model returns a microbatch-mean classification loss.
        # Let Trainer apply gradient-accumulation scaling.
        trainer.model_accepts_loss_kwargs = False

        checkpoint_dir = run_dir / "checkpoints"

        checkpoint_paths = (
            list(checkpoint_dir.glob("checkpoint-*"))
            if checkpoint_dir.exists()
            else []
        )

        required_checkpoint_files = [
            "trainer_state.json",
            "adapter_config.json",
            "adapter_model.safetensors",
            "optimizer.pt",
            "scheduler.pt",
            "rng_state.pth",
        ]

        valid_checkpoints = [
            path
            for path in checkpoint_paths
            if all(
                (path / name).is_file()
                for name in required_checkpoint_files
            )
        ]

        last_checkpoint = (
            max(
                valid_checkpoints,
                key=lambda path: int(
                    path.name.split("-")[-1]
                ),
            )
            if valid_checkpoints
            else None
        )

        print(
            "Expected optimizer steps:",
            optimizer_steps,
        )
        print("Resume checkpoint:", last_checkpoint)

        began = time.time()

        training_result = trainer.train(
            resume_from_checkpoint=(
                str(last_checkpoint)
                if last_checkpoint
                else None
            )
        )

        model = trainer.model

        adapter_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            str(adapter_dir),
            safe_serialization=True,
            save_embedding_layers=False,
        )

        tokenizer.save_pretrained(str(adapter_dir))

        with safe_open(
            str(adapter_dir / "adapter_model.safetensors"),
            framework="pt",
            device="cpu",
        ) as adapter_file:
            if not any(
                "score" in name
                for name in adapter_file.keys()
            ):
                raise ValueError(
                    "Saved adapter is missing its "
                    "trained classification head."
                )

        completed = {
            "spec_signature": specification_signature,
            "adapter_sha256": file_hash(
                adapter_dir / "adapter_model.safetensors"
            ),
            "base_revision": revision,
            "best_checkpoint": (
                trainer.state.best_model_checkpoint
            ),
            "best_sample_macro_f1": (
                trainer.state.best_metric
            ),
            "training_metrics": training_result.metrics,
            "training_minutes_this_invocation": (
                time.time() - began
            ) / 60,
            "log_history": trainer.state.log_history,
        }

        save_json(complete_path, completed)

        save_json(
            output_dir / "cwq_training_metrics_v1.json",
            completed,
        )

    controller_configuration = {
        "spec_signature": specification_signature,
        "base_model": MODEL_NAME,
        "base_revision": revision,
        "class_to_id": CLASS_TO_ID,
        "maximum_sequence_length": MAX_LENGTH,
        "num_labels": 4,
        "padding_side": "right",
        "truncation_side": "right",
        "frontier_limit": configuration["frontier_limit"],
        "max_actions": configuration["action_cap"],
        "max_hops": configuration["max_hops"],
        "controller_text_version": (
            configuration["controller_text_version"]
        ),
        "quantization": "4-bit NF4",
        "data_signature": generation["signature"],
    }

    save_json(
        adapter_dir / "controller_config.json",
        controller_configuration,
    )

    model.eval()

    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    def predict_texts(texts):
        probabilities = np.empty(
            (len(texts), 4),
            dtype=np.float32,
        )

        # Sort by observed input length and restore original order.
        order = sorted(
            range(len(texts)),
            key=lambda index: len(texts[index]),
        )

        with torch.inference_mode():
            for start in range(
                0,
                len(order),
                INFERENCE_BATCH,
            ):
                indices = order[
                    start:start + INFERENCE_BATCH
                ]

                encoded = tokenizer(
                    [texts[index] for index in indices],
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    pad_to_multiple_of=8,
                    return_tensors="pt",
                )

                encoded = {
                    name: tensor.to("cuda")
                    for name, tensor in encoded.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = model(
                        **encoded,
                        return_dict=True,
                    ).logits

                values = torch.softmax(
                    logits.float(),
                    dim=-1,
                ).cpu().numpy()

                probabilities[indices] = values

        return probabilities

    probe = predict_texts(
        [item["text"] for item in dev_sample[:4]]
    )

    if (
        probe.shape != (min(4, len(dev_sample)), 4)
        or not np.isfinite(probe).all()
    ):
        raise RuntimeError(
            "Trained-controller inference check failed."
        )

    adapter_identity = fingerprint({
        name: file_hash(adapter_dir / name)
        for name in [
            "adapter_model.safetensors",
            "adapter_config.json",
            "controller_config.json",
        ]
    })

    prediction_signature = fingerprint({
        "adapter_identity": adapter_identity,
        "dev_sha256": generation["splits"]["dev"]["sha256"],
        "max_length": MAX_LENGTH,
        "chunk_states": PREDICTION_CHUNK_STATES,
        "inference_batch": INFERENCE_BATCH,
        "code_version": CODE_VERSION,
    })

    print(
        "\nScoring ALL development states and candidates "
        "for global STOP calibration..."
    )
    print(
        "Completed prediction chunks are saved to Drive "
        "and reused if this cell is rerun."
    )

    candidate_metrics, state_records = complete_dev_predictions(
        state_paths["dev"],
        generation["splits"]["dev"],
        allowed_qids["dev"],
        output_dir,
        prediction_signature,
        predict_texts,
    )

    candidate_path = (
        output_dir / "cwq_dev_candidate_metrics_v1.json"
    )

    save_json(
        candidate_path,
        {
            "prediction_signature": prediction_signature,
            **candidate_metrics,
        },
    )

    state_path = (
        output_dir / "cwq_dev_state_predictions_v1.jsonl"
    )

    temporary = state_path.with_name(
        state_path.name + ".partial"
    )

    with open(temporary, "w", encoding="utf-8") as file:
        for record in state_records:
            file.write(json.dumps(record) + "\n")

    temporary.replace(state_path)

    best, sweep = calibrate(state_records)

    proxy = np.array([
        record["stop_score_proxy"]
        for record in state_records
    ])

    truth = np.array([
        record["global_stop_label"]
        for record in state_records
    ])

    hidden_states = [
        record
        for record in state_records
        if record["shortlist_hides_all_positive"]
    ]

    threshold_report = {
        "selection_split": "development",
        "selected_threshold": best["threshold"],
        "decision_rule": (
            "STOP if candidate_count == 0 or "
            "max(1 - P(gain_0)) < threshold"
        ),
        "target": (
            "full-frontier stop_label, "
            "not shortlist_stop_label"
        ),
        "selection_rule": (
            "Maximize STOP F1; then minimize conditional "
            "premature stopping; then maximize accuracy; "
            "then smaller threshold."
        ),
        "selected_threshold_metrics": best,
        "development_states": len(state_records),
        "development_questions": len(allowed_qids["dev"]),
        "development_candidates": (
            candidate_metrics["number_of_examples"]
        ),
        "shortlist_hidden_states": len(hidden_states),
        "hidden_state_metrics": (
            stop_metrics(
                hidden_states,
                best["threshold"],
            )
            if hidden_states
            else None
        ),
        "stop_score_proxy_brier": float(
            np.mean(np.square(proxy - truth))
        ),
        "proxy_note": (
            "1 - max P(gain>0) is a diagnostic score, "
            "not a calibrated joint probability of "
            "answer-set completeness."
        ),
        "answer_recognition": (
            "Oracle during offline state generation; "
            "online learned-recognition evaluation "
            "is still required."
        ),
        "frontier_limit": configuration["frontier_limit"],
        "max_actions": configuration["action_cap"],
        "max_hops": configuration["max_hops"],
        "model_revision": revision,
        "adapter_identity": adapter_identity,
        "prediction_signature": prediction_signature,
        "test_used": False,
        "threshold_sweep": sweep,
    }

    threshold_path = (
        output_dir / "cwq_dev_stop_threshold_v1.json"
    )

    save_json(threshold_path, threshold_report)

    summary = {
        "model": MODEL_NAME,
        "base_revision": revision,
        "adapter_path": str(adapter_dir),
        "data_signature": generation["signature"],
        "train_questions": len(allowed_qids["train"]),
        "development_questions": len(allowed_qids["dev"]),
        "training_examples": len(train_sample),
        "checkpoint_dev_examples": len(dev_sample),
        "full_dev_candidates": (
            candidate_metrics["number_of_examples"]
        ),
        "full_dev_states": len(state_records),
        "maximum_sequence_length": MAX_LENGTH,
        "frontier_limit": configuration["frontier_limit"],
        "max_actions": configuration["action_cap"],
        "max_hops": configuration["max_hops"],
        "candidate_accuracy": candidate_metrics["accuracy"],
        "candidate_balanced_accuracy": (
            candidate_metrics["balanced_accuracy"]
        ),
        "candidate_macro_f1": candidate_metrics["macro_f1"],
        "majority_class_accuracy": (
            candidate_metrics["majority_class_accuracy"]
        ),
        "selected_stop_threshold": best["threshold"],
        "selected_stop_metrics": best,
        "shortlist_hidden_states": len(hidden_states),
        "transformers_version": transformers.__version__,
        "peft_version": peft.__version__,
        "pytorch_version": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "training_minutes_this_invocation": (
            completed["training_minutes_this_invocation"]
        ),
        "sampling_train": train_stats,
        "sampling_checkpoint_dev": dev_stats,
        "dependency_versions": {
            package: importlib.metadata.version(package)
            for package in [
                "transformers",
                "peft",
                "accelerate",
                "bitsandbytes",
                "safetensors",
            ]
        },
        "base_training_budget": (
            "60,000 balanced candidates, one epoch, "
            "rank-8 LoRA on q/k/v/o"
        ),
        "test_used": False,
        "next_step": (
            "Train a separate CWQ answer-node recognizer "
            "before end-to-end evaluation."
        ),
    }

    summary_path = (
        output_dir / "cwq_controller_training_summary_v1.json"
    )

    save_json(summary_path, summary)

    inventory_paths = [
        summary_path,
        candidate_path,
        threshold_path,
        state_path,
        adapter_dir / "controller_config.json",
        adapter_dir / "adapter_config.json",
        adapter_dir / "adapter_model.safetensors",
    ]

    output_inventory_path = (
        output_dir / "artifact_inventory_v1.json"
    )

    save_json(
        output_inventory_path,
        {
            "data_signature": generation["signature"],
            "files": [
                {
                    "name": path.name,
                    "path": str(path),
                    "bytes": path.stat().st_size,
                    "sha256": file_hash(path),
                }
                for path in inventory_paths
            ],
        },
    )

    print("\n" + "=" * 90)
    print("CWQ CONTROLLER TRAINING AND DEV CALIBRATION COMPLETE")
    print("=" * 90)
    print(json.dumps(summary, indent=2))

    print("\nSaved CWQ adapter:")
    print(adapter_dir)

    print("\nSEND THESE FOUR REPORTS:")

    for path in [
        summary_path,
        candidate_path,
        threshold_path,
        output_inventory_path,
    ]:
        print(path)


main()

Mounted at /content/drive
NOTEBOOK 13: CWQ QWEN GAIN CONTROLLER TRAINING
GPU: Tesla T4
Transformers: 5.17.0
PyTorch: 2.11.0+cu130
PEFT: 0.21.0
Frontier limit: 128
Action cap: 24
Maximum hops: 4
Verifying state file: cwq_train_states_v1_full.jsonl
Verifying state file: cwq_dev_states_v1_full.jsonl
train: 5,000 states / 75,579 candidates scanned
train: 10,000 states / 150,763 candidates scanned
train: 15,000 states / 225,847 candidates scanned
train: 20,000 states / 301,240 candidates scanned
train: 25,000 states / 376,283 candidates scanned
train: 30,000 states / 451,437 candidates scanned
train: 35,000 states / 526,848 candidates scanned
train: 40,000 states / 601,872 candidates scanned
train: 45,000 states / 677,659 candidates scanned
train: 50,000 states / 752,815 candidates scanned
train: 55,000 states / 827,792 candidates scanned
train: 60,000 states / 903,209 candidates scanned
train: 65,000 states / 978,419 candidates scanned
train: 70,000 states / 1,054,047 candidates scanned
tr

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

trainable params: 1,084,928 || all params: 495,121,280 || trainable%: 0.2191
Unsupported optional arguments omitted: ['save_safetensors']
Expected optimizer steps: 1875
Resume checkpoint: None


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
500,1.093705,1.279798,0.501000,0.500369,0.373308,0.556712
1000,0.896801,0.816039,0.663250,0.567461,0.485797,0.704144
1500,0.691663,0.718968,0.723625,0.617502,0.547649,0.754072
1875,0.675467,0.746797,0.713625,0.631635,0.550395,0.750060


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]


Scoring ALL development states and candidates for global STOP calibration...
Completed prediction chunks are saved to Drive and reused if this cell is rerun.
Full dev: 512/9,625 states | 45,271 candidates | 7.1 min
Full dev: 1,024/9,625 states | 86,351 candidates | 13.5 min
Full dev: 1,536/9,625 states | 128,710 candidates | 19.9 min
Full dev: 2,048/9,625 states | 173,309 candidates | 27.0 min
Full dev: 2,560/9,625 states | 218,349 candidates | 34.1 min
Full dev: 3,072/9,625 states | 267,484 candidates | 42.1 min
Full dev: 3,584/9,625 states | 313,689 candidates | 49.1 min
Full dev: 4,096/9,625 states | 363,889 candidates | 56.9 min
Full dev: 4,608/9,625 states | 409,930 candidates | 64.4 min
Full dev: 5,120/9,625 states | 452,315 candidates | 70.9 min
Full dev: 5,632/9,625 states | 500,274 candidates | 78.4 min
Full dev: 6,144/9,625 states | 544,554 candidates | 85.3 min
Full dev: 6,656/9,625 states | 591,632 candidates | 92.5 min
Full dev: 7,168/9,625 states | 637,533 candidates | 9